# Day 5 — Meeting Minutes Generator (Gradio UI)  
**Combining Frontier (Whisper API) & Open-Source LLMs**  
This notebook turns an audio recording into polished **meeting minutes** with **discussion points**, **takeaways**, and **action items**—and wraps it in a **Gradio UI**.

### What you’ll learn / use
- Audio → text transcription with either **OpenAI Whisper API** (frontier) or **local Whisper** (open-source).
- Meeting-minutes summarization using an **open‑source instruct model** (default: `microsoft/Phi-3-mini-4k-instruct`) with **4‑bit quantization** when a GPU is present.
- A simple, production‑style **Gradio** app with streaming‑like UX.

## 0) Runtime check (GPU, Python)

In [1]:
import os, platform, sys, subprocess, shutil, json, time

print("Python:", sys.version)
print("Platform:", platform.platform())
print("CUDA visible devices:", os.environ.get("CUDA_VISIBLE_DEVICES"))
try:
    import torch
    print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available(), "| Devices:", torch.cuda.device_count())
    if torch.cuda.is_available():
        print("CUDA device:", torch.cuda.get_device_name(0))
except Exception as e:
    print("Torch not yet installed:", e)

Python: 3.12.11 (main, Jun  4 2025, 08:56:18) [GCC 11.4.0]
Platform: Linux-6.1.123+-x86_64-with-glibc2.35
CUDA visible devices: None
PyTorch: 2.8.0+cu126 | CUDA: True | Devices: 1
CUDA device: NVIDIA A100-SXM4-40GB


## 1) Installs
- Core: `transformers`, `accelerate`, `bitsandbytes`, `sentencepiece`
- UI: `gradio`
- Audio: `faster-whisper` (fast local ASR) and/or `transformers` Whisper pipeline option
- Optional: `openai` for Whisper API

In [2]:
# For Colab you may wish to pin CUDA wheels, but here we keep general installs.
# If you are on Colab with a CUDA GPU and want the compiled PyTorch wheel, uncomment the next line and adapt the CUDA version.
# !pip install -q --upgrade torch torchvision torchaudio

!pip install -q -U transformers accelerate bitsandbytes sentencepiece gradio pandas
!pip install -q -U faster-whisper
!pip install -q -U openai==1.*

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 116.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 138.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.2 which is incompatible.
dask-cudf-cu12 25.6.0 requires pandas<2.2.4dev0,>=2.0, but you have pandas 2.3.2 which is incompatible.
cudf-cu12 25.6.0 requires pandas<2.2.4dev0,>=2.0, but you have pandas 2.3.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.9/39.9 

## 2) Imports

In [3]:
import os, re, json, math, tempfile, pathlib, threading, queue
from dataclasses import dataclass
from typing import List, Dict, Any, Optional

import torch
import pandas as pd
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TextIteratorStreamer
)
import gradio as gr

# Optional OpenAI client for Whisper API
try:
    from openai import OpenAI
except Exception:
    OpenAI = None

# Optional local ASR options
try:
    from faster_whisper import WhisperModel as FasterWhisperModel
    _HAS_FASTER = True
except Exception:
    _HAS_FASTER = False

# For a transformers-based Whisper pipeline (heavier)
try:
    from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline as hf_pipeline
    _HAS_HF_WHISPER = True
except Exception:
    _HAS_HF_WHISPER = False

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

## 3) (Optional) Google Drive mount for Colab users

In [4]:
# If you're in Google Colab and want to pull audio from Drive, mount it here.
try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
    print("Drive mounted at /content/drive")
except Exception:
    print("Not running in Colab or drive not available; skipping mount.")

Not running in Colab or drive not available; skipping mount.


## 4) Auth & model choices
- Set **`OPENAI_API_KEY`** in your environment if you plan to use **Whisper API** for transcription.
- Optionally set **`HUGGINGFACE_TOKEN`** if you need gated models.

In [5]:
from huggingface_hub import login

HF_TOKEN = os.environ.get("HUGGINGFACE_TOKEN", None)
if HF_TOKEN:
    try:
        login(token=HF_TOKEN, add_to_git_credential=False)
        print("✅ Hugging Face login successful.")
    except Exception as e:
        print("⚠️ HF login failed:", e)
else:
    print("ℹ️ No HUGGINGFACE_TOKEN found. Proceeding unauthenticated (okay for most models).")

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", None)
if OPENAI_API_KEY and OpenAI is not None:
    try:
        openai_client = OpenAI(api_key=OPENAI_API_KEY)
        print("✅ OpenAI client configured.")
    except Exception as e:
        print("⚠️ Could not init OpenAI client:", e)
        openai_client = None
else:
    openai_client = None
    print("ℹ️ No OPENAI_API_KEY found or OpenAI lib not available. Whisper API disabled.")

ℹ️ No HUGGINGFACE_TOKEN found. Proceeding unauthenticated (okay for most models).
ℹ️ No OPENAI_API_KEY found or OpenAI lib not available. Whisper API disabled.


## 5) Load an open‑source summarization model (4‑bit when possible)
Default: `microsoft/Phi-3-mini-4k-instruct` (strong, lightweight).  
You can switch to a larger instruct model if you have a stronger GPU (e.g., Qwen2.5‑7B‑Instruct).

In [6]:
DEFAULT_SUMMARIZER = "microsoft/Phi-3-mini-4k-instruct"
SUMMARIZER_OPTIONS = [
    "microsoft/Phi-3-mini-4k-instruct",
    "Qwen/Qwen2.5-7B-Instruct",
    "HuggingFaceH4/zephyr-7b-beta",
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
]

def load_summarizer(model_name: str):
    print(f"Loading summarizer: {model_name}")
    kwargs = {}
    bnb = None
    if DEVICE == "cuda":
        try:
            bnb = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_use_double_quant=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float16,
            )
            kwargs["quantization_config"] = bnb
            kwargs["device_map"] = "auto"
            kwargs["torch_dtype"] = torch.bfloat16 if torch.cuda.is_available() else torch.float16
        except Exception as e:
            print("Quantization not available, loading normally:", e)
            kwargs["device_map"] = "auto" if DEVICE == "cuda" else None
    else:
        kwargs["device_map"] = None

    tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    mdl = AutoModelForCausalLM.from_pretrained(model_name, **kwargs)
    return tok, mdl

tokenizer, model = load_summarizer(DEFAULT_SUMMARIZER)

Loading summarizer: microsoft/Phi-3-mini-4k-instruct


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.67G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

## 6) Transcription helpers (choose one of three paths)
- **OpenAI Whisper API** (frontier) – easiest/robust if you have an API key.
- **Local ASR: faster‑whisper** – fast and good quality on modern GPUs/CPUs.
- **Local ASR: transformers Whisper pipeline** – heavier but portable.

In [7]:
def transcribe_openai(audio_path: str) -> str:
    if openai_client is None:
        raise RuntimeError("OpenAI client not configured or API key missing.")
    with open(audio_path, "rb") as f:
        # Whisper-1 remains widely available. You can also try 'gpt-4o-mini-transcribe' if enabled.
        result = openai_client.audio.transcriptions.create(model="whisper-1", file=f, response_format="text")
    return result

def transcribe_faster_whisper(audio_path: str, model_size: str = "medium") -> str:
    if not _HAS_FASTER:
        raise RuntimeError("faster-whisper not installed.")
    compute_type = "float16" if DEVICE == "cuda" else "int8"
    asr = FasterWhisperModel(model_size, device=DEVICE, compute_type=compute_type)
    segments, info = asr.transcribe(audio_path, beam_size=5)
    text = " ".join(s.text for s in segments)
    return text.strip()

def transcribe_hf_whisper(audio_path: str, model_id: str = "openai/whisper-medium") -> str:
    if not _HAS_HF_WHISPER:
        raise RuntimeError("Transformers whisper pipeline not available.")
    asr_model = AutoModelForSpeechSeq2Seq.from_pretrained(
        model_id, torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
        low_cpu_mem_usage=True, use_safetensors=True
    )
    if DEVICE == "cuda":
        asr_model.to("cuda")
    proc = AutoProcessor.from_pretrained(model_id)
    pipe = hf_pipeline(
        "automatic-speech-recognition", model=asr_model,
        tokenizer=proc.tokenizer, feature_extractor=proc.feature_extractor,
        device=0 if DEVICE == "cuda" else -1, return_timestamps=False
    )
    result = pipe(audio_path)
    return result["text"]

## 7) Minutes prompt + generation

In [9]:
SYSTEM_PROMPT = (
        "You are an expert note‑taker who writes professional minutes of meetings. "
        "Return Markdown with sections: Summary (with attendees, location, date if present), "
        "Discussion Points, Takeaways, and Action Items (with owners and due dates if mentioned). "
        "Be concise, faithful to the transcript, and avoid adding facts not supported by the text."
    )

def build_messages(transcript: str, extra_instructions: str = ""):
    user = (
        "Below is a transcript of a meeting. Write polished minutes in Markdown with:\n"
        "- Summary (include attendees, location, date if present)\n"
        "- Discussion Points\n"
        "- Takeaways\n"
        "- Action Items (with owners and due dates if mentioned)\n\n"
        f"{extra_instructions}\n\n"
        "### Transcript\n"
        f"{transcript}\n"
    )
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user},
    ]
    return messages

def generate_minutes(transcript: str, temperature: float = 0.4, top_p: float = 0.9, max_new_tokens: int = 1200):
    messages = build_messages(transcript)
    input_ids = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt")
    if DEVICE == "cuda":
        input_ids = input_ids.to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            input_ids=input_ids,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    # Drop the prompt part
    gen_ids = output_ids[0, input_ids.shape[1]:]
    text = tokenizer.decode(gen_ids, skip_special_tokens=True)
    return text.strip()

## 8) Gradio App
- Upload an audio file **or** paste a transcript.
- Choose transcription method.
- Choose summarizer model (optional).

In [10]:
def ensure_model(selected:str):
    global tokenizer, model
    if selected and selected != getattr(model, 'name_or_path', None):
        tokenizer, model = load_summarizer(selected)
    return selected

def pipeline_run(audio_file, transcript_text, transcribe_method, summarizer_model, temperature, top_p):
    # Optionally reload summarizer if the user changed it
    ensure_model(summarizer_model)

    # 1) Get transcript
    if transcript_text and transcript_text.strip():
        transcript = transcript_text.strip()
    else:
        if audio_file is None:
            return "⚠️ Please upload an audio file or paste a transcript.", None
        audio_path = audio_file if isinstance(audio_file, str) else audio_file.name
        if transcribe_method == "OpenAI Whisper API":
            if openai_client is None:
                return "⚠️ OPENAI_API_KEY missing or OpenAI client not available.", None
            transcript = transcribe_openai(audio_path)
        elif transcribe_method == "faster-whisper (local)":
            transcript = transcribe_faster_whisper(audio_path, model_size="medium")
        else:
            transcript = transcribe_hf_whisper(audio_path, model_id="openai/whisper-medium")

    # 2) Summarize
    minutes_md = generate_minutes(transcript, temperature=temperature, top_p=top_p, max_new_tokens=1200)

    # 3) Save minutes to a temp file for download
    tmpdir = tempfile.mkdtemp()
    out_md = os.path.join(tmpdir, "meeting_minutes.md")
    with open(out_md, "w", encoding="utf-8") as f:
        f.write(minutes_md)

    return minutes_md, out_md

with gr.Blocks(title="Meeting Minutes Generator") as demo:
    gr.Markdown("## Meeting Minutes Generator — Audio → Transcript → Minutes (Markdown)")
    with gr.Row():
        with gr.Column():
            audio = gr.Audio(label="Upload audio (mp3/wav/m4a)", type="filepath")
            transcript_text = gr.Textbox(label="Or paste transcript (optional)", lines=8, placeholder="Paste transcript here to skip ASR...")
            transcribe_method = gr.Radio(
                label="Transcription method",
                value="faster-whisper (local)",
                choices=["OpenAI Whisper API", "faster-whisper (local)", "transformers Whisper (local)"]
            )
            summarizer_model = gr.Dropdown(
                label="Summarizer (open-source)",
                value=DEFAULT_SUMMARIZER,
                choices=SUMMARIZER_OPTIONS
            )
            temperature = gr.Slider(0.0, 1.5, value=0.4, step=0.05, label="Temperature")
            top_p = gr.Slider(0.1, 1.0, value=0.9, step=0.05, label="top_p")
            btn = gr.Button("Generate Minutes", variant="primary")
        with gr.Column():
            minutes = gr.Markdown(label="Minutes (Markdown)")
            out_file = gr.File(label="Download minutes")

    btn.click(
        fn=pipeline_run,
        inputs=[audio, transcript_text, transcribe_method, summarizer_model, temperature, top_p],
        outputs=[minutes, out_file],
    )

print("✅ App ready. In Colab, call demo.launch(share=True) to get a public link.")

✅ App ready. In Colab, call demo.launch(share=True) to get a public link.


### Launch the app (Colab or local Jupyter)

In [12]:
# Uncomment to run in-notebook.
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://483c25957340ef7fb5.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
